In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, HTMLMath
from IPython.display import display

# ------------------------------------------------------------
# INTRODUCTION
# ------------------------------------------------------------

intro_html = HTML("""
<div style="font-size:14px; line-height:1.55; width:1100px; padding:8px 12px; margin-bottom:10px;">
<b>Interactive Pole-Zero Geometry of a Biquadratic Second-Order Filter</b><br><br>

This notebook illustrates how the natural frequencies
<b>ω<sub>0p</sub></b>, <b>ω<sub>0z</sub></b> and the quality factors
<b>Q<sub>p</sub></b>, <b>Q<sub>z</sub></b> determine the locations of the
complex-conjugate poles and zeros of a second-order biquadratic transfer function.
The quality factors are restricted to values greater than 0.5, so that both
the poles and zeros remain complex-conjugate pairs.<br><br>

Move the sliders and observe how increasing Q moves the corresponding pair
closer to the imaginary axis, while changing ω<sub>0</sub> changes its radial
distance from the origin. The zeros may also be placed either in the left or
the right half-plane.<br><br>

The two concentric circles represent the constant-magnitude loci of the poles
and zeros. The pole pair always lies on the circle with radius
<b>|p| = ω<sub>0p</sub></b>, whereas the zero pair lies on the circle with radius
<b>|z| = ω<sub>0z</sub></b>. Therefore, varying Q<sub>p</sub> or Q<sub>z</sub>
moves the corresponding pair along its circle, while varying ω<sub>0p</sub> or
ω<sub>0z</sub> changes the radius of that circle.
</div>
""")

# ------------------------------------------------------------
# PARAMETER CONTROLS
# ------------------------------------------------------------

wp_title = HTML(value="<b>Pole Natural Frequency ω<sub>0p</sub></b>")
wp_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=2.5, description='', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='240px'))

qp_title = HTML(value="<b>Pole Quality Factor Q<sub>p</sub></b>")
qp_slider = FloatSlider(min=0.51, max=5.0, step=0.05, value=1.0, description='', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='240px'))

wz_title = HTML(value="<b>Zero Natural Frequency ω<sub>0z</sub></b>")
wz_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=2.0, description='', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='240px'))

qz_title = HTML(value="<b>Zero Quality Factor Q<sub>z</sub></b>")
qz_slider = FloatSlider(min=0.51, max=5.0, step=0.05, value=1.2, description='', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='240px'))

wp_box = VBox([wp_title, wp_slider], layout=Layout(width='255px', overflow='visible'))
qp_box = VBox([qp_title, qp_slider], layout=Layout(width='255px', overflow='visible'))
wz_box = VBox([wz_title, wz_slider], layout=Layout(width='255px', overflow='visible'))
qz_box = VBox([qz_title, qz_slider], layout=Layout(width='255px', overflow='visible'))

parameter_controls = HBox([wp_box, qp_box, wz_box, qz_box], layout=Layout(width='1080px', justify_content='space-between', align_items='flex-start', overflow='visible'))

# ------------------------------------------------------------
# ZERO LOCATION CONTROLS
# ------------------------------------------------------------

zero_side_title = HTML(value="<b>Zero Location:</b>", layout=Layout(width='95px', overflow='visible'))

zero_side_radio = RadioButtons(options=['Left half-plane', 'Right half-plane'], value='Left half-plane', description='', layout=Layout(width='270px', overflow='visible'))

# ------------------------------------------------------------
# SYMBOLIC EQUATIONS
# ------------------------------------------------------------

pole_equation_output = HTMLMath(layout=Layout(width='360px', overflow='visible'))
zero_equation_output = HTMLMath(layout=Layout(width='360px', overflow='visible'))

# ------------------------------------------------------------
# CSS
# ------------------------------------------------------------

display(HTML("""
<style>

.horizontal-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    align-items: center !important;
    gap: 16px !important;
    overflow: visible !important;
}

.horizontal-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
    display: flex !important;
    align-items: center !important;
}

.horizontal-radio > label {
    display: none !important;
}

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.output_scroll {
    max-height: none !important;
    height: auto !important;
    overflow: visible !important;
    overflow-y: visible !important;
    overflow-x: visible !important;
}

</style>
"""))

zero_side_radio.add_class('horizontal-radio')

# ------------------------------------------------------------
# COMPACT INFORMATION ROW
# ------------------------------------------------------------

info_row = HBox([zero_side_title, zero_side_radio, pole_equation_output, zero_equation_output], layout=Layout(width='1120px', align_items='center', justify_content='flex-start', overflow='visible'))

controls = VBox([parameter_controls, info_row], layout=Layout(width='1120px', margin='0 0 8px 0', overflow='visible'))

# ------------------------------------------------------------
# EXTRA SPACE BEFORE THE FIGURE
# ------------------------------------------------------------

figure_spacer = HTML("<div style='height:28px;'></div>")

# ------------------------------------------------------------
# MAIN INTERACTIVE FUNCTION
# ------------------------------------------------------------

def plot_pole_zero_geometry(w0p, Qp, w0z, Qz, zero_side):

    # --------------------------------------------------------
    # POLES
    # --------------------------------------------------------

    pole_real = -w0p / (2.0 * Qp)
    pole_imag = w0p * np.sqrt(1.0 - 1.0 / (4.0 * Qp**2))

    p1 = pole_real + 1j * pole_imag
    p2 = pole_real - 1j * pole_imag

    # --------------------------------------------------------
    # ZEROS
    # --------------------------------------------------------

    zero_sign = -1.0 if zero_side == 'Left half-plane' else 1.0

    zero_real = zero_sign * w0z / (2.0 * Qz)
    zero_imag = w0z * np.sqrt(1.0 - 1.0 / (4.0 * Qz**2))

    z1 = zero_real + 1j * zero_imag
    z2 = zero_real - 1j * zero_imag

    # --------------------------------------------------------
    # SYMBOLIC EQUATIONS
    # --------------------------------------------------------

    zero_sign_tex = "-" if zero_side == 'Left half-plane' else "+"

    pole_equation_output.value = rf"$$p_{{1,2}}=-\frac{{\omega_{{0p}}}}{{2Q_p}}\pm j\omega_{{0p}}\sqrt{{1-\frac{{1}}{{4Q_p^2}}}}$$"

    zero_equation_output.value = rf"$$z_{{1,2}}={zero_sign_tex}\frac{{\omega_{{0z}}}}{{2Q_z}}\pm j\omega_{{0z}}\sqrt{{1-\frac{{1}}{{4Q_z^2}}}}$$"

    # --------------------------------------------------------
    # FIGURE
    # ------------------------------------------------------------

    fig, ax = plt.subplots(figsize=(9.2, 9.2))

    # --------------------------------------------------------
    # REFERENCE CIRCLES
    # ------------------------------------------------------------

    theta = np.linspace(0.0, 2.0 * np.pi, 800)

    pole_circle_x = w0p * np.cos(theta)
    pole_circle_y = w0p * np.sin(theta)

    zero_circle_x = w0z * np.cos(theta)
    zero_circle_y = w0z * np.sin(theta)

    ax.plot(pole_circle_x, pole_circle_y, linestyle='--', linewidth=1.3, alpha=0.65, label=r'$|p|=\omega_{0p}$')
    ax.plot(zero_circle_x, zero_circle_y, linestyle=':', linewidth=1.3, alpha=0.65, label=r'$|z|=\omega_{0z}$')

    # --------------------------------------------------------
    # POLE PROJECTIONS
    # ------------------------------------------------------------

    ax.plot([pole_real, pole_real], [-pole_imag, pole_imag], linestyle='--', linewidth=1.2, alpha=0.75)
    ax.plot([pole_real, 0.0], [pole_imag, pole_imag], linestyle='--', linewidth=1.2, alpha=0.75)
    ax.plot([pole_real, 0.0], [-pole_imag, -pole_imag], linestyle='--', linewidth=1.2, alpha=0.75)

    # --------------------------------------------------------
    # ZERO PROJECTIONS
    # ------------------------------------------------------------

    ax.plot([zero_real, zero_real], [-zero_imag, zero_imag], linestyle='--', linewidth=1.2, alpha=0.75)
    ax.plot([0.0, zero_real], [zero_imag, zero_imag], linestyle='--', linewidth=1.2, alpha=0.75)
    ax.plot([0.0, zero_real], [-zero_imag, -zero_imag], linestyle='--', linewidth=1.2, alpha=0.75)

    # --------------------------------------------------------
    # POLES
    # ------------------------------------------------------------

    ax.scatter([np.real(p1), np.real(p2)], [np.imag(p1), np.imag(p2)], marker='x', s=130, linewidths=2.6, label='Poles')

    # --------------------------------------------------------
    # ZEROS
    # ------------------------------------------------------------

    ax.scatter([np.real(z1), np.real(z2)], [np.imag(z1), np.imag(z2)], marker='o', s=95, color='red', edgecolors='red', linewidths=1.5, label='Zeros')

    # --------------------------------------------------------
    # RADIAL LINES
    # ------------------------------------------------------------

    ax.plot([0.0, pole_real], [0.0, pole_imag], linewidth=1.2, alpha=0.65)
    ax.plot([0.0, pole_real], [0.0, -pole_imag], linewidth=1.2, alpha=0.65)

    ax.plot([0.0, zero_real], [0.0, zero_imag], linewidth=1.2, alpha=0.65)
    ax.plot([0.0, zero_real], [0.0, -zero_imag], linewidth=1.2, alpha=0.65)

    # --------------------------------------------------------
    # AXES
    # ------------------------------------------------------------

    ax.axhline(0.0, linewidth=1.2)
    ax.axvline(0.0, linewidth=1.2)

    limit = 1.12 * max(w0p, w0z)

    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)

    ax.set_aspect('equal', adjustable='box')

    ax.set_title('Pole-Zero Geometry of a Biquadratic Second-Order Filter', fontsize=12)
    ax.set_xlabel(r'Real Axis $\sigma$', fontsize=11)
    ax.set_ylabel(r'Imaginary Axis $j\omega$', fontsize=11)

    ax.grid(True, linestyle=':', alpha=0.55)

    # --------------------------------------------------------
    # LEGEND
    # ------------------------------------------------------------

    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0, fontsize=9, labelspacing=0.9)

    # --------------------------------------------------------
    # CURRENT POLE AND ZERO COORDINATES
    # ------------------------------------------------------------

    ax.text(1.03, 0.70, 'Current coordinates', transform=ax.transAxes, fontsize=9, fontweight='bold', va='top')

    ax.text(1.03, 0.65, rf'$p_{{1,2}}={pole_real:.3f}\pm j\,{pole_imag:.3f}$', transform=ax.transAxes, fontsize=9, va='top')

    ax.text(1.03, 0.59, rf'$z_{{1,2}}={zero_real:.3f}\pm j\,{zero_imag:.3f}$', transform=ax.transAxes, fontsize=9, va='top')

    # --------------------------------------------------------
    # FORMAT
    # ------------------------------------------------------------

    ax.tick_params(axis='x', labelsize=9)
    ax.tick_params(axis='y', labelsize=9)

    fig.subplots_adjust(left=0.09, right=0.74, bottom=0.09, top=0.93)

    plt.show()
    plt.close(fig)

# ------------------------------------------------------------
# INTERACTIVE WIDGET
# ------------------------------------------------------------

widget_plot = interactive(plot_pole_zero_geometry, w0p=wp_slider, Qp=qp_slider, w0z=wz_slider, Qz=qz_slider, zero_side=zero_side_radio)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

plot_output.layout.height = 'auto'
plot_output.layout.max_height = 'none'
plot_output.layout.overflow = 'visible'

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

display(intro_html)
display(controls)
display(figure_spacer)
display(plot_output)